In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your actual folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_I.xpt"
csv_file = DATA_DIR / "SMQFAM_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Keep an untouched copy of the pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 SMQFAM_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD460",
    "SMD470",
    "SMD480"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQFAM_I columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9971:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 4:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,971 rows x 4 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached SMQFAM_I:
#
#     genuine decimal variables = NONE
#     string variables          = NONE
#
#
# Variables:
#
# SEQN
#     Respondent identifier
#
# SMD460
#     Number of people in household who smoke
#
# SMD470
#     Number of people who smoke inside home
#
# SMD480
#     Number of days during past week someone
#     smoked inside home
#
#
# All released values are mathematically whole-number.
#
# Therefore all 4 fields can safely become nullable Int64
# AFTER the validated tiny-zero restoration.
#
#
# IMPORTANT:
#
# This conclusion applies ONLY to SMQFAM_I.
#
# It does NOT change your broader NHANES rule:
#
#     Dietary genuine decimals
#     Laboratory continuous values
#     Examination measurements
#     survey weights
#
# remain exactly AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 4:

    raise ValueError(
        "Expected exactly 4 whole-number fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# NEVER globally convert this value to zero
# until the exact variables/counts have been validated.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 4 SMQFAM_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{ordinary_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached SMQFAM_I.xpt:
#
#     SMD460    6,930
#     SMD470    1,769
#     SMD480       99
#
# TOTAL       = 8,798
#
#
# CDC confirms:
#
# SMD460:
#     0 = No one in household is a smoker
#     Count = 6,930
#
# SMD470:
#     0 = No one smokes inside house
#     Count = 1,769
#
# SMD480:
#     0 = zero days during past week
#     Count = 99
#

EXPECTED_TINY_COUNTS = {
    "SMD460": 6930,
    "SMD470": 1769,
    "SMD480": 99
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated SMQFAM_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 8798:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 3:

    raise ValueError(
        "Expected tiny values in exactly 3 variables."
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before tiny-value repair."
    )


print(
    "\nPASS: Exact 8,798-value tiny-zero pattern "
    "across 3 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT do:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only variables whose pattern was validated.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 8798:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df becomes the exact fidelity reference.
#
# ONLY:
#
#     validated tiny value -> numeric zero
#
# may differ from source_df.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Do this BEFORE Int64 conversion.
#
# np.round() here is ONLY used to detect whether
# fractional values exist.
#
# It NEVER overwrites source values.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected decimal variables found.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in SMQFAM_I."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 12. PROVE ALL 4 FIELDS ARE WHOLE-NUMBER
# ============================================================

whole_number_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        whole_number_failures.append(
            col
        )


if whole_number_failures:

    raise ValueError(
        "Unexpected fractional values found in:\n"
        + str(
            whole_number_failures
        )
    )


print(
    "PASS: All 4 SMQFAM_I fields are "
    "mathematically whole-number."
)


# ============================================================
# 13. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 4 fields converted to nullable Int64 "
    "without rounding."
)


# ============================================================
# 14. CDC CHECK - SMD460
# ============================================================

#
# Number of people living here who smoke tobacco.
#
# CDC:
#
#     0   No one                    6,930
#     1   1 household smoker       1,803
#     2   2 household smokers        710
#     3   3 or more smokers          200
#     777 Refused                      4
#     999 Don't know                   2
#
# Missing = 322
#

EXPECTED_SMD460 = {
    0: 6930,
    1: 1803,
    2: 710,
    3: 200,
    777: 4,
    999: 2
}


actual_smd460 = (
    df["SMD460"]
    .value_counts()
    .sort_index()
    .to_dict()
)


smd460_missing = int(
    df["SMD460"]
    .isna()
    .sum()
)


print(
    "\n--- SMD460 ---"
)

print(
    actual_smd460
)

print(
    "Missing:",
    smd460_missing
)


if actual_smd460 != EXPECTED_SMD460:

    raise ValueError(
        "SMD460 frequencies differ.\n\n"
        f"Expected:\n{EXPECTED_SMD460}\n\n"
        f"Actual:\n{actual_smd460}"
    )


if smd460_missing != 322:

    raise ValueError(
        "SMD460 missing count differs."
    )


print(
    "PASS: SMD460 matches CDC exactly."
)


# ============================================================
# 15. CDC CHECK - SMD470
# ============================================================

#
# Number of people who smoke inside this home.
#
# CDC:
#
#     0   No one                    1,769
#     1   1 person                    590
#     2   2 people                    280
#     3   3 or more                    74
#
# Missing = 7,258
#

EXPECTED_SMD470 = {
    0: 1769,
    1: 590,
    2: 280,
    3: 74
}


actual_smd470 = (
    df["SMD470"]
    .value_counts()
    .sort_index()
    .to_dict()
)


smd470_missing = int(
    df["SMD470"]
    .isna()
    .sum()
)


print(
    "\n--- SMD470 ---"
)

print(
    actual_smd470
)

print(
    "Missing:",
    f"{smd470_missing:,}"
)


if actual_smd470 != EXPECTED_SMD470:

    raise ValueError(
        "SMD470 frequencies differ.\n\n"
        f"Expected:\n{EXPECTED_SMD470}\n\n"
        f"Actual:\n{actual_smd470}"
    )


if smd470_missing != 7258:

    raise ValueError(
        "SMD470 missing count differs."
    )


print(
    "PASS: SMD470 matches CDC exactly."
)


# ============================================================
# 16. SMD460 -> SMD470 ROUTING CHECK
# ============================================================

#
# CDC:
#
# If SMD460 is:
#
#     0
#     DK
#     RF
#
# then go to end of section.
#
#
# SMD470 therefore appears exactly when SMD460 is:
#
#     1
#     2
#     3
#
#
# Attachment:
#
#     SMD460 1-3 = 2,713
#     SMD470 nonmissing = 2,713
#

smd470_expected = (
    df["SMD460"]
    .between(
        1,
        3
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smd470_present = (
    df["SMD470"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smd470_routing_mismatch = int(
    np.sum(
        smd470_expected
        !=
        smd470_present
    )
)


print(
    "\n--- SMD460 / SMD470 ROUTING ---"
)

print(
    "SMD460 positive smoker households:",
    int(
        smd470_expected.sum()
    )
)

print(
    "SMD470 populated:",
    int(
        smd470_present.sum()
    )
)

print(
    "Routing mismatches:",
    smd470_routing_mismatch
)


if int(
    smd470_expected.sum()
) != 2713:

    raise ValueError(
        "Unexpected SMD470 eligibility count."
    )


if smd470_routing_mismatch != 0:

    raise ValueError(
        "SMD470 availability differs "
        "from SMD460 routing."
    )


print(
    "PASS: SMD460/SMD470 routing validated."
)


# ============================================================
# 17. SMD470 <= SMD460 CONSISTENCY CHECK
# ============================================================

#
# CDC specifies:
#
# SMD470 cannot be greater than SMD460.
#
#
# Code value 3 represents 3 or more in both variables,
# so numeric comparison remains valid for the released
# categories.
#

both_present = (
    df["SMD460"].notna()
    &
    df["SMD470"].notna()
)


smd470_gt_460 = int(
    (
        both_present
        &
        (
            df["SMD470"]
            >
            df["SMD460"]
        )
    )
    .sum()
)


print(
    "\nSMD470 > SMD460 violations:",
    smd470_gt_460
)


if smd470_gt_460 != 0:

    raise ValueError(
        "SMD470 exceeds SMD460 "
        "for one or more records."
    )


print(
    "PASS: SMD470 <= SMD460 consistency preserved."
)


# ============================================================
# 18. CDC CHECK - SMD480
# ============================================================

#
# Days during past week somebody smoked inside home.
#
# CDC:
#
#     0 = 99
#     1 = 100
#     2 = 100
#     3 = 56
#     4 = 15
#     5 = 25
#     6 = 12
#     7 = 530
#     99 = Don't know = 7
#
# Missing = 9,027
#

EXPECTED_SMD480 = {
    0: 99,
    1: 100,
    2: 100,
    3: 56,
    4: 15,
    5: 25,
    6: 12,
    7: 530,
    99: 7
}


actual_smd480 = (
    df["SMD480"]
    .value_counts()
    .sort_index()
    .to_dict()
)


smd480_missing = int(
    df["SMD480"]
    .isna()
    .sum()
)


print(
    "\n--- SMD480 ---"
)

print(
    actual_smd480
)

print(
    "Missing:",
    f"{smd480_missing:,}"
)


if actual_smd480 != EXPECTED_SMD480:

    raise ValueError(
        "SMD480 frequencies differ.\n\n"
        f"Expected:\n{EXPECTED_SMD480}\n\n"
        f"Actual:\n{actual_smd480}"
    )


if smd480_missing != 9027:

    raise ValueError(
        "SMD480 missing count differs."
    )


print(
    "PASS: SMD480 matches CDC exactly."
)


# ============================================================
# 19. SMD470 -> SMD480 ROUTING CHECK
# ============================================================

#
# CDC:
#
# SMD470:
#
#     If 0 -> End of section
#
# Therefore SMD480 is asked when SMD470 indicates
# one or more people smoke inside:
#
#     1
#     2
#     3
#
#
# Attachment:
#
#     SMD470 1-3 = 944
#     SMD480 nonmissing = 944
#

smd480_expected = (
    df["SMD470"]
    .between(
        1,
        3
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smd480_present = (
    df["SMD480"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smd480_routing_mismatch = int(
    np.sum(
        smd480_expected
        !=
        smd480_present
    )
)


print(
    "\n--- SMD470 / SMD480 ROUTING ---"
)

print(
    "SMD470 positive inside-smoker households:",
    int(
        smd480_expected.sum()
    )
)

print(
    "SMD480 populated:",
    int(
        smd480_present.sum()
    )
)

print(
    "Routing mismatches:",
    smd480_routing_mismatch
)


if int(
    smd480_expected.sum()
) != 944:

    raise ValueError(
        "Unexpected SMD480 eligibility count."
    )


if smd480_routing_mismatch != 0:

    raise ValueError(
        "SMD480 availability differs "
        "from SMD470 routing."
    )


print(
    "PASS: SMD470/SMD480 routing validated."
)


# ============================================================
# 20. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded XPT:
#
#     ordinary zeros = 0
#
#
# Therefore every actual zero after repair must match
# exactly one of the validated tiny-value positions.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 8798:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 8,798 validated restorations."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ONLY:
#
#     validated tiny -> zero
#
# changes are permitted.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 4 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# SMQFAM_I contains no true decimals.
#
# But using this same safe export approach maintains
# consistency with your Dietary/Lab/Exam workflow.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 23. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 24. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 4 SMQFAM_I variables."
)


# ============================================================
# 25. MISSING-VALUE FIDELITY
# ============================================================

missing_position_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 26. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 27. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 28. RAW CSV .0 CHECK
# ============================================================

#
# SMQFAM_I contains no genuine continuous decimal fields.
#
# Therefore it is safe to check ALL 4 columns for
# unwanted ".0".
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- RAW CSV .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any SMQFAM_I field."
)


# ============================================================
# 29. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 8,798 validated zero values."
)


# ============================================================
# 30. FINAL FREQUENCY CHECK - SMD460
# ============================================================

final_smd460 = (
    pd.to_numeric(
        raw_csv_df["SMD460"]
        .where(
            raw_csv_df["SMD460"] != "",
            np.nan
        ),
        errors="coerce"
    )
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


if final_smd460 != EXPECTED_SMD460:

    raise ValueError(
        "Final SMD460 frequencies changed."
    )


print(
    "PASS: Final SMD460 frequencies preserved."
)


# ============================================================
# 31. FINAL FREQUENCY CHECK - SMD470
# ============================================================

final_smd470 = (
    pd.to_numeric(
        raw_csv_df["SMD470"]
        .where(
            raw_csv_df["SMD470"] != "",
            np.nan
        ),
        errors="coerce"
    )
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


if final_smd470 != EXPECTED_SMD470:

    raise ValueError(
        "Final SMD470 frequencies changed."
    )


print(
    "PASS: Final SMD470 frequencies preserved."
)


# ============================================================
# 32. FINAL FREQUENCY CHECK - SMD480
# ============================================================

final_smd480 = (
    pd.to_numeric(
        raw_csv_df["SMD480"]
        .where(
            raw_csv_df["SMD480"] != "",
            np.nan
        ),
        errors="coerce"
    )
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


if final_smd480 != EXPECTED_SMD480:

    raise ValueError(
        "Final SMD480 frequencies changed."
    )


print(
    "PASS: Final SMD480 frequencies preserved."
)


# ============================================================
# 33. FINAL ROUTING CHECKS
# ============================================================

final_460 = pd.to_numeric(
    raw_csv_df["SMD460"]
    .where(
        raw_csv_df["SMD460"] != "",
        np.nan
    ),
    errors="coerce"
)


final_470 = pd.to_numeric(
    raw_csv_df["SMD470"]
    .where(
        raw_csv_df["SMD470"] != "",
        np.nan
    ),
    errors="coerce"
)


expected_470_present = (
    final_460
    .between(
        1,
        3
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


actual_470_present = (
    raw_csv_df["SMD470"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_470_mismatch = int(
    np.sum(
        expected_470_present
        !=
        actual_470_present
    )
)


expected_480_present = (
    final_470
    .between(
        1,
        3
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


actual_480_present = (
    raw_csv_df["SMD480"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_480_mismatch = int(
    np.sum(
        expected_480_present
        !=
        actual_480_present
    )
)


print(
    "\n--- FINAL ROUTING CHECK ---"
)

print(
    "SMD460/SMD470 mismatches:",
    final_470_mismatch
)

print(
    "SMD470/SMD480 mismatches:",
    final_480_mismatch
)


if final_470_mismatch != 0:

    raise ValueError(
        "Final SMD460/SMD470 routing changed."
    )


if final_480_mismatch != 0:

    raise ValueError(
        "Final SMD470/SMD480 routing changed."
    )


print(
    "PASS: Final routing relationships preserved."
)


# ============================================================
# 34. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 35. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "SMQFAM_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - "
    "Smoking / Household Smokers"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "SMD460 restored zeros:",
    int(
        df["SMD460"]
        .eq(0)
        .sum()
    )
)


print(
    "SMD470 restored zeros:",
    int(
        df["SMD470"]
        .eq(0)
        .sum()
    )
)


print(
    "SMD480 restored zeros:",
    int(
        df["SMD480"]
        .eq(0)
        .sum()
    )
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "SMD460/SMD470 routing mismatches:",
    smd470_routing_mismatch
)


print(
    "SMD470/SMD480 routing mismatches:",
    smd480_routing_mismatch
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "SMQFAM_I contains no genuine decimal variables."
)


print(
    "Only the 8,798 validated XPORT tiny values "
    "are changed to real numeric zero."
)


print(
    "All 4 fields can then safely be exported "
    "as whole-number values using nullable Int64."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "variables remain exactly AS IS."
)

NHANES 2015-2016 SMQFAM_I XPT -> CSV
Rows: 9,971
Columns: 4
PASS: Structure = 9,971 rows x 4 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number fields: ['SEQN', 'SMD460', 'SMD470', 'SMD480']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 8,798
Variables containing tiny artifacts: 3

Tiny values by variable:
SMD460    6930
SMD470    1769
SMD480      99

PASS: Exact 8,798-value tiny-zero pattern across 3 variables validated.

--- VALIDATED ZERO RESTORATION ---
SMD460: 6,930 restored to 0
SMD470: 1,769 restored to 0
SMD480: 99 restored to 0

PASS: 8,798 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
SMD460: 0 fractional observations
SMD470: 0 fractional observations
SMD480: 0 fractional observations

PASS: No genuine decimal variables found in SMQFAM_I.

--- SEQN ---
Minimum: 83732
Maximum